In [1]:
import pandas as pd
import os
import numpy as np
import sklearn
import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, PreTrainedTokenizer, TrainingArguments, Trainer
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from dataclasses import dataclass, field
import transformers
from peft import (
    LoraConfig,
    get_peft_model,
    get_peft_model_state_dict,
)
import json

c:\Users\Sai\Documents\GitHub\SBOLtrainer\dna-bert\dnabertvenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
current_dir = os.path.abspath('')
data_path = os.path.join(current_dir, '..', 'seqtrainer-data')
dataset_path = os.path.join(current_dir, '..', 'hpc', "hpc_datasets")
output_dir = os.path.join(current_dir, '..', 'dnabert-out')

In [6]:
df = pd.read_csv(os.path.join(dataset_path, "hpc_seq_target_data.csv"))

In [7]:
df

,sequence,target
0,GCAAATTTTGCACAAAAAATAGGCTTTAGTGATTTGTTTTTGTTCA...,52.010547
1,GTATCTGCCTCCGATTCTCTGCAGAAGCAGAAAGACATTGGATCGA...,0.565387
2,CTTCCAGGCGGGTGGGGTCAATGTCCATCAGGGCAATATGCGCCGT...,0.690511
3,ATCAAAAATGAAGCCGATAACGGCCTGCGCAACACGCGTGGCACCA...,0.696464
4,TTCTTCATCGGGCAGATCTTCAAACGGTACCAGTCGGCGGACGTTT...,0.561674
...,...,...
17050,CTTTCTGCGCACCATCACCATCGAGACGAGCGATACCGTACTCCAG...,0.727274
17051,GGGCGCAGAAACAGCTTTGCTTACTGGAACATAACGACGCATGACG...,0.517912
17052,CGAATGCTGTTTTTTTAATCACACCTTTATCCTTTCGCTGTCTTGC...,0.909039
17053,AGGTCGAACCGGAAATCCAGGTCAGCACTTGCGCCATTCGCGGGTC...,0.586090


In [8]:
@dataclass
class DataCollatorForSupervisedDataset(object):
    """Collate examples for supervised fine-tuning."""

    tokenizer: transformers.PreTrainedTokenizer

    def __call__(self, instances):
        input_ids, labels = tuple([instance[key] for instance in instances] for key in ("input_ids", "labels"))
        input_ids = torch.nn.utils.rnn.pad_sequence(
            input_ids, batch_first=True, padding_value=self.tokenizer.pad_token_id
        )
        labels = torch.tensor(labels, dtype=torch.float32)
        return dict(
            input_ids=input_ids,
            labels=labels,
            attention_mask=input_ids.ne(self.tokenizer.pad_token_id),
        )


In [9]:
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]
    preds = preds.squeeze() 
    mse = ((preds - labels) ** 2).mean()
    return {"mse": mse, "rmse": mse**0.5}

In [10]:


class SupervisedDataset(Dataset):
    """Dataset for supervised fine-tuning."""

    def __init__(self, sequences, tokenizer, labels):

        super(SupervisedDataset, self).__init__()
        output = tokenizer(
            sequences,
            return_tensors="pt",
            padding="longest",
            max_length=4096,
            truncation=True,
        )

        self.input_ids = output["input_ids"]
        self.attention_mask = output["attention_mask"]
        self.labels = torch.tensor(labels, dtype=torch.float32)
        self.num_labels = len(set(labels))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, i):
        return dict(input_ids=self.input_ids[i], labels=self.labels[i])

In [11]:
X = df['sequence']
y = np.log1p(np.array(df['target']))

In [12]:
# train, test, val
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

In [13]:
tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True)
collator = DataCollatorForSupervisedDataset(tokenizer=tokenizer)

In [14]:
max(y_train)

np.float64(5.123536442341581)

In [15]:
train_dataset = SupervisedDataset(tokenizer=tokenizer, sequences=X_train.tolist(), labels=y_train.tolist())
val_dataset = SupervisedDataset(tokenizer=tokenizer, sequences=X_val.tolist(), labels=y_val.tolist())
test_dataset = SupervisedDataset(tokenizer=tokenizer, sequences=X_test.tolist(), labels=y_test.tolist())

In [16]:
model = AutoModelForSequenceClassification.from_pretrained(
    "zhihan1996/DNABERT-2-117M",
    num_labels=1,
    problem_type="regression", 
    trust_remote_code=True
)

target_modules = ["Wqkv", "dense", "gated_layers"]

# configure LoRA
lora_config = LoraConfig(r=8,lora_alpha=32, target_modules=target_modules,lora_dropout=0.05, bias="none", task_type="SEQ_CLS",inference_mode=False)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

model.config.return_dict = True
model.config.output_hidden_states = False
model.config.output_attentions = False


training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=1e-3,                   
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    logging_steps=100,
    gradient_accumulation_steps=2,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=True,                          
    load_best_model_at_end=True,
    metric_for_best_model="rmse",
    greater_is_better=False,
)


# define trainer
trainer = transformers.Trainer(model=model,
                        tokenizer=tokenizer,
                        args=training_args,
                        compute_metrics=compute_metrics,
                        train_dataset=train_dataset,
                        eval_dataset=val_dataset,
                        data_collator=collator)
trainer.train()


C:\Users\Sai\.cache\huggingface\modules\transformers_modules\zhihan1996\DNABERT-2-117M\7bce263b15377fc15361f52cfab88f8b586abda0\bert_layers.py:126: UserWarning: Unable to import Triton; defaulting MosaicBERT attention implementation to pytorch (this will reduce throughput when using this model).
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at zhihan1996/DNABERT-2-117M and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\Sai\AppData\Local\Temp\ipykernel_27476\1727249749.py:42: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = transformers.Trainer(model=model,
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and genera

trainable params: 1,118,977 || all params: 118,188,290 || trainable%: 0.9468


c:\Users\Sai\Documents\GitHub\SBOLtrainer\dna-bert\dnabertvenv\lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [1]:
from transformers import Trainer, TrainingArguments, AutoModelForSequenceClassification, AutoTokenizer
def model_init(trial=None):
    return AutoModelForSequenceClassification.from_pretrained(
        "zhihan1996/DNABERT-2-117M",
        num_labels=1,
        problem_type="regression", 
        trust_remote_code=True)

def optuna_hp_space(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True),
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [8, 16, 32]),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
        "warmup_ratio": trial.suggest_float("warmup_ratio", 0.0, 0.2),
        "optimizer": trial.suggest_categorical("optimizer", ["adamw_torch", "adamw_hf", "adafactor"]),
    }


training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=5,
    evaluation_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    logging_steps=100,
    gradient_accumulation_steps=2,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="rmse",
    greater_is_better=False,
)

trainer = transformers.Trainer(
                        model_init=model_init,
                        tokenizer=tokenizer,
                        args=training_args,
                        compute_metrics=compute_metrics,
                        train_dataset=train_dataset,
                        eval_dataset=val_dataset,
                        data_collator=collator)

best_trial = trainer.hyperparameter_search(
    direction="minimize",
    backend="optuna",
    hp_space=optuna_hp_space,
    n_trials=50,
)

trainer.apply_hyperparameters(best_trial.hyperparameters)
trainer.train() 

metrics = trainer.evaluate()
print(metrics)


KeyboardInterrupt: 

In [ ]:
results_path = os.path.join(training_args.output_dir, "results")
results = trainer.evaluate(eval_dataset=test_dataset)
os.makedirs(results_path, exist_ok=True)
with open(os.path.join(results_path, "eval_results.json"), "w") as f:
    json.dump(results, f)

In [ ]:
print(model)


In [ ]:
#class DNABertTuned(torch.nn.Module):
#     def __init__(self):
#         super(DNABertTuned, self).__init__()
#         # self.regression = nn.Linear(768, 1)
#         self.bert = AutoModel.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True)
#         self.regression_head = torch.nn.Linear(self.bert.config.hidden_size, 1) 

#     def forward(self, input_ids, attention_mask=None):
#         outputs = self.bert(input_ids, attention_mask=attention_mask)[0]
#         pooled_output = torch.mean(outputs, dim=1)
#         regression_output = self.regression_head(pooled_output)
#         return regression_output.squeeze(-1)

# model = DNABertTuned()
# model.train()

# optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

# seqs = list(df['sequence'])
# inputs = tokenizer(seqs, return_tensors = 'pt', padding=True)
# output = model(inputs['input_ids'], attention_mask=inputs['attention_mask'])
# print(output.shape)
# print(target.shape)
# loss = torch.nn.MSELoss(target, output)
# optimizer.zero_grad()
# loss.backward()
# optimizer.step()
